# Qwen 3B / 1.5B Reconciliation Pipeline

Unified successor to `qwen3b_full_stack.ipynb` and the (now-retired) `qwen_past_200_h100.ipynb`. One notebook that does the work that has to happen on a GPU host:

1. **Cell 1** — setup, deps, repo, knobs.
2. **Cell 2** — build/resume the corpus, run AWQ per-tensor + AWQ per-channel calibrations, dump the frozen baselines for 1.5B and 3B, and emit the Q2/IQ2 importance artifact for the future sub-2-bit ship.
3. **Cell 3** — Eagle6 training. A **short grid sweep** to find the per-target winner, then **one extra-long training session** on that winner with full corpus and longer windows. This is the lever the user explicitly asked for.
4. **Cell 4** — Tau ranking + frontier policy search across long-trained heads, plus an **in-notebook spec-decode simulation** that estimates real accept-per-verify against held-out corpus — so we know whether the head is good *before* the Mac runtime port lands.
5. **Cell 5** — Summary + the **local Rust port plan** the next session will execute. (Spec-decode is currently wired into `deepseek_v2.rs` only; Qwen-3B/1.5B dense path does not call the Eagle head at runtime. The trained heads in this notebook are inventory waiting on that port — they are *not* the bottleneck.)

What's preserved from the retired past-200 notebook (legacy under `colab/legacy/`): variable hidden, `--num-blocks`, head-heads/ff-mult, 2-block training family, Q2/IQ2 importance, 1.5B student path.

What's new in Eagle6:
- Adaptive per-channel AWQ (`colab/awq_per_channel_calibrate.py`) layered on top of the per-tensor smoothing.
- Extra-long training pass on the grid winner (20+ epochs, full corpus, `--max-row-tokens 192-256`, residual-delta + calib weight tuned).
- In-notebook acceptance simulation against held-out corpus, so the projection isn't multipliers-and-hope.

Resumable: every long-running step skips if its artifact is already on Drive.

In [ ]:
# Cell 1 - Setup, deps, repo, knobs
from google.colab import drive
drive.mount('/content/drive')

import glob, json, math, os, shutil, subprocess, sys, time
from pathlib import Path


def _unbuffered_cmd(cmd):
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    return cmd

def run(cmd, **kwargs):
    cmd = _unbuffered_cmd(cmd)
    env = os.environ.copy()
    env.update(kwargs.pop('env', {}) or {})
    env['PYTHONUNBUFFERED'] = '1'
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=True, env=env, **kwargs)

run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
    'hf_transfer>=0.1.9',
])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTHONUNBUFFERED', '1')
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = None
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets.')
else:
    print('No HF_TOKEN secret found; public Hub downloads still work.')

import numpy as np
import torch
import transformers

assert torch.cuda.is_available(), 'No CUDA: set Runtime -> Change runtime type -> GPU'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')
print(f'transformers={transformers.__version__} torch={torch.__version__}')

# ---- Idle-disconnect keep-alive -----------------------------------------
# Colab browser tabs drop the runtime connection if the cell output goes
# silent for too long (~90s on Pro, ~30 min hard idle limit). When that
# happens the cell shows "Reconnecting" and you can't see what the
# subprocess is doing — even though the backend is often still running.
# This JS snippet clicks the connect button every 60s so the runtime
# stays attached. Works in the actual Colab UI; no-op in classic Jupyter.
try:
    from IPython.display import Javascript, display
    display(Javascript('''
function dismantle_keep_alive(){
    try {
        const btn = document.querySelector("colab-toolbar-button#connect")
                 || document.querySelector("colab-connect-button")
                 || document.querySelector("colab-toolbar-button[command='connect']");
        if (btn) { btn.click(); }
    } catch(e) { /* tab may be inactive; ignore */ }
}
if (window.__dismantle_keepalive_handle) {
    clearInterval(window.__dismantle_keepalive_handle);
}
window.__dismantle_keepalive_handle = setInterval(dismantle_keep_alive, 60000);
console.log("[dismantle] Colab keep-alive installed (clicks connect every 60s)");
'''))
    print('Installed Colab keep-alive (clicks connect button every 60s)')
except Exception as e:
    print(f'(keep-alive injection skipped: {e})')

# Heartbeat helper for cells that spawn long subprocesses — printing every
# ~45s keeps Colab from treating the cell as idle even if the child process
# is silent (e.g. mega_calibrate during a large shard read).
def _heartbeat_loop(stop_evt, interval=45.0, label='heartbeat'):
    import threading, time
    while not stop_evt.is_set():
        if stop_evt.wait(interval):
            return
        print(f'[{label}] still alive @ {time.strftime("%H:%M:%S")}', flush=True)

def start_heartbeat(label='cell2'):
    """Returns (stop_event, thread). Call stop_event.set() to stop."""
    import threading
    stop_evt = threading.Event()
    t = threading.Thread(target=_heartbeat_loop, args=(stop_evt, 45.0, label), daemon=True)
    t.start()
    return stop_evt, t

# ---- Master knobs --------------------------------------------------------
MAX_QUALITY_MODE = True             # full corpus + long-train pass
RUN_SHORT_GRID = True               # 5-spec grid to pick the winner
RUN_LONG_TRAIN = True               # extra-long single-spec training on winner
RUN_SIMULATION = True               # in-notebook spec-decode accept estimation
FORCE_CORPUS = False
FORCE_AWQ = False
FORCE_FROZEN = False
FORCE_RETRAIN = False
FORCE_TAU = False
FORCE_FRONTIER = False
USE_TORCH_COMPILE = False

# ---- Models we train heads for ------------------------------------------
TARGETS = [
    {'tag': 'q3b',  'model_id': 'Qwen/Qwen2.5-3B-Instruct',   'enabled': True},
    {'tag': 'q1p5', 'model_id': 'Qwen/Qwen2.5-1.5B-Instruct', 'enabled': True},
]
CAPTURE_LAYER_Q3B = 32
CAPTURE_LAYER_Q15 = 22   # Qwen2.5-1.5B has fewer layers; pick a late but not final block

BASELINE_DEC_TPS_Q3B = 27.0
BASELINE_DEC_TPS_Q15 = 55.0
W4A8_MULTIPLIER = 1.15      # conservative after the May 2026 local bench dose-of-reality
SPEC_EFFICIENCY = 0.70      # Mac-side realistic, not 0.85 Colab assumption

# ---- Paths ---------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/dismantle'
ARTIFACT_DIR = f'{DRIVE_ROOT}/qwen_reconciliation'
CORPUS_DIR_Q3B = f'{DRIVE_ROOT}/qwen3b_corpus'
CORPUS_DIR_Q15 = f'{DRIVE_ROOT}/qwen1p5_corpus'
LOCAL_FROZEN_DIR = '/content/frozen_local'

# ---- Corpus + training sizing -------------------------------------------
CORPUS_TARGET_SEQS = 10_000 if MAX_QUALITY_MODE else 3_000

# Grid pass: 5 specs × 5 epochs, batch=16. Discovers winner per target.
GRID_EPOCHS = 5
GRID_MAX_ROWS = 4_000
GRID_MAX_ROW_TOKENS = 128

# Long-train pass: winner gets 20 epochs, longer windows, full corpus.
LONG_EPOCHS = 20
LONG_MAX_ROWS = 8_000
LONG_MAX_ROW_TOKENS = 192

EVAL_MAX_WINDOWS = 6_000 if MAX_QUALITY_MODE else 2_000
FRONTIER_MAX_DEPTH = 12
FRONTIER_TOP_N = 1   # only the long-trained winner per target

# ---- Eagle6 training spec grid (5 specs) --------------------------------
# Mix of 1-block + 2-block, mix of LRs, mix of residual-delta weights.
EAGLE6_GRID = [
    {'family': 'e6_b1_base', 'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 4.0,
     'lr': 1e-3, 'residual_delta': 0.00, 'calib_weight': 0.10},
    {'family': 'e6_b1_wide', 'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 6.0,
     'lr': 6e-4, 'residual_delta': 0.00, 'calib_weight': 0.10},
    {'family': 'e6_b2_lite', 'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 4.0,
     'lr': 6e-4, 'residual_delta': 0.01, 'calib_weight': 0.15},
    {'family': 'e6_b2_hot',  'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 4.0,
     'lr': 1e-3, 'residual_delta': 0.02, 'calib_weight': 0.20},
    {'family': 'e6_b2_wide', 'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 6.0,
     'lr': 6e-4, 'residual_delta': 0.02, 'calib_weight': 0.20},
]

# Long-train pass uses the same head architecture as the winner spec but
# overrides LR/residual_delta/calib_weight from a tuned recipe per target.
# These are the 'extra-long training session' hyperparameters.
LONG_TRAIN_OVERRIDES = {
    'lr': 5e-4,                # lower LR, longer schedule
    'residual_delta': 0.02,
    'calib_weight': 0.20,
}

# ---- Working dirs --------------------------------------------------------
for d in (DRIVE_ROOT, ARTIFACT_DIR, CORPUS_DIR_Q3B, CORPUS_DIR_Q15,
          f'{ARTIFACT_DIR}/checkpoints', LOCAL_FROZEN_DIR):
    os.makedirs(d, exist_ok=True)

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'
if not os.path.exists('/content/dismantle'):
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/dismantle'])
else:
    run(['git', '-C', '/content/dismantle', 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', '/content/dismantle', 'checkout', BRANCH])
    run(['git', '-C', '/content/dismantle', 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir('/content/dismantle')
print('repo:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

REQUIRED = [
    'colab/mega_calibrate.py',
    'colab/eagle5_train_pytorch.py',
    'colab/eagle5_tau_eval_pytorch.py',
    'colab/eagle5_frontier_policy.py',
    'colab/build_qwen3b_frozen_hf.py',
    'colab/q2k_importance_calibrate.py',
    'tools/training/awq_calibrate.py',
    'colab/awq_per_channel_calibrate.py',
]
for path in REQUIRED:
    assert os.path.exists(path), f'missing required file: {path}'
    print('ok', path)

print(json.dumps({
    'MAX_QUALITY_MODE': MAX_QUALITY_MODE,
    'TARGETS': [t['tag'] for t in TARGETS if t['enabled']],
    'CORPUS_TARGET_SEQS': CORPUS_TARGET_SEQS,
    'GRID_EPOCHS': GRID_EPOCHS, 'LONG_EPOCHS': LONG_EPOCHS,
    'GRID_MAX_ROWS': GRID_MAX_ROWS, 'LONG_MAX_ROWS': LONG_MAX_ROWS,
    'GRID_MAX_ROW_TOKENS': GRID_MAX_ROW_TOKENS,
    'LONG_MAX_ROW_TOKENS': LONG_MAX_ROW_TOKENS,
    'EAGLE6_GRID': [s['family'] for s in EAGLE6_GRID],
    'ARTIFACT_DIR': ARTIFACT_DIR,
}, indent=2))


In [ ]:
# Cell 2 - Corpus + AWQ (per-tensor + per-channel) + frozen baselines + Q2/IQ2 importance.
# Per-target. The 1.5B and 3B share the same Hugging Face dataset stream but
# different capture layers + frozen weight dumps.
#
# Heartbeat thread prints every ~45s so Colab's idle-disconnect heuristic
# doesn't drop the runtime mid-build. (`start_heartbeat` defined in Cell 1.)
import glob, os, json, math, subprocess, sys, time, shutil
import numpy as np

_hb_stop, _hb_thread = start_heartbeat(label='cell2-calibration')

# If a fresh calibration produces no local OR Drive shard inside this window,
# stop it and keep the already-finished primary target moving. This catches HF
# download/socket stalls and dataset-stream stalls that otherwise look alive only
# because the parent heartbeat still prints.
CALIBRATION_FIRST_SHARD_TIMEOUT_SEC = 20 * 60
CALIBRATION_STATUS_INTERVAL_SEC = 60
SKIP_OPTIONAL_TARGET_ON_CALIBRATION_STALL = True
OPTIONAL_TARGETS = {'q1p5'}

# Accept a partial corpus as "complete enough" if we have at least 60% of
# the target sequences (floor 2000). Cell 3's training consumes max-rows
# worth of windows, not the full corpus — beyond ~4000 seqs the AWQ outlier
# stats stabilize and additional sequences mostly add Drive cost. This
# also rescues partial runs where mega_calibrate errored late: if you have
# 6300+ seqs on Drive, no need to relaunch the entire build.
SHIPPABLE_FRACTION = 0.60
def _shippable_threshold(target):
    return max(2000, int(target * SHIPPABLE_FRACTION))

def stats_sequences(stats_path):
    if not os.path.exists(stats_path):
        return 0
    try:
        with np.load(stats_path) as z:
            return int(np.asarray(z.get('sequences_written', 0)).item())
    except Exception:
        return 0

def corpus_state(corpus_dir, target_seqs):
    shards = len(glob.glob(f'{corpus_dir}/shard_*.parquet'))
    stats = f'{corpus_dir}/per_site_activation_stats.npz'
    seqs = stats_sequences(stats)
    required_full = math.ceil(target_seqs / 16)
    shippable = _shippable_threshold(target_seqs)
    complete_full = shards >= required_full and seqs >= target_seqs
    complete_shippable = seqs >= shippable
    return {
        'shards': shards, 'seqs': seqs,
        'required_shards_full': required_full,
        'shippable_seqs': shippable,
        'complete_full': complete_full,
        'complete_shippable': complete_shippable,
        'stats': stats,
    }

def shard_count(path):
    return len(glob.glob(f'{path}/shard_*.parquet'))

def unbuffered_python_cmd(cmd):
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    return cmd

def run_calibration(cmd, *, tag, local_out, sync_dir):
    cmd = unbuffered_python_cmd(cmd)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('$ ' + ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, env=env)
    start = time.time()
    last_status = 0.0
    try:
        while True:
            rc = proc.poll()
            now = time.time()
            elapsed = now - start
            local_shards = shard_count(local_out)
            drive_shards = shard_count(sync_dir)
            seqs = max(
                stats_sequences(f'{local_out}/per_site_activation_stats.npz'),
                stats_sequences(f'{sync_dir}/per_site_activation_stats.npz'),
            )
            if now - last_status >= CALIBRATION_STATUS_INTERVAL_SEC:
                print(
                    f'[cal-watch:{tag}] elapsed={elapsed/60:.1f}m '
                    f'local_shards={local_shards} drive_shards={drive_shards} '
                    f'stats_seqs={seqs}',
                    flush=True,
                )
                last_status = now
            if rc is not None:
                if rc != 0:
                    raise subprocess.CalledProcessError(rc, cmd)
                print(f'[cal-watch:{tag}] finished cleanly', flush=True)
                return
            if local_shards + drive_shards == 0 and elapsed >= CALIBRATION_FIRST_SHARD_TIMEOUT_SEC:
                print(
                    f'[cal-watch:{tag}] no shards after {elapsed/60:.1f}m; terminating calibration',
                    flush=True,
                )
                proc.terminate()
                try:
                    proc.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
                raise TimeoutError(
                    f'{tag} calibration produced no shards in '
                    f'{CALIBRATION_FIRST_SHARD_TIMEOUT_SEC // 60} minutes'
                )
            time.sleep(5)
    except Exception:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        raise

if VRAM_GB >= 70:
    tier, load_4bit, batch, lm_head_chunk = 'H100 80GB+', False, 8, 256
elif VRAM_GB >= 35:
    tier, load_4bit, batch, lm_head_chunk = 'A100 40GB-class', False, 6, 128
elif VRAM_GB >= 20:
    tier, load_4bit, batch, lm_head_chunk = 'L4 24GB-class', True, 4, 64
else:
    tier, load_4bit, batch, lm_head_chunk = 'T4/V100 16GB', True, 2, 32
print(f'GPU tier: {tier}  batch={batch}  load={"4bit" if load_4bit else "fp16"}')

TARGET_META = {
    'q3b':  {'model_id': 'Qwen/Qwen2.5-3B-Instruct',   'capture_layer': CAPTURE_LAYER_Q3B,
             'corpus_dir': CORPUS_DIR_Q3B,
             'frozen': f'{ARTIFACT_DIR}/qwen3b_frozen.npz',
             'awq_tensor': f'{ARTIFACT_DIR}/qwen3b_awq.json',
             'awq_channel': f'{ARTIFACT_DIR}/qwen3b_awq_per_channel.json',
             'q2_importance': f'{ARTIFACT_DIR}/qwen3b_q2_importance.npz'},
    'q1p5': {'model_id': 'Qwen/Qwen2.5-1.5B-Instruct', 'capture_layer': CAPTURE_LAYER_Q15,
             'corpus_dir': CORPUS_DIR_Q15,
             'frozen': f'{ARTIFACT_DIR}/qwen1p5_frozen.npz',
             'awq_tensor': f'{ARTIFACT_DIR}/qwen1p5_awq.json',
             'awq_channel': f'{ARTIFACT_DIR}/qwen1p5_awq_per_channel.json',
             'q2_importance': f'{ARTIFACT_DIR}/qwen1p5_q2_importance.npz'},
}

try:
    for t in TARGETS:
        if not t['enabled']:
            continue
        tag = t['tag']
        meta = TARGET_META[tag]
        stats_path = f'{meta["corpus_dir"]}/per_site_activation_stats.npz'
        meta['stats'] = stats_path
        print(f'\n=== Calibration for {tag} ({meta["model_id"]}) ===')
        state = corpus_state(meta['corpus_dir'], CORPUS_TARGET_SEQS)
        print('  corpus state:', {k: v for k, v in state.items() if k != 'stats'})
        if state['complete_full'] and not FORCE_CORPUS:
            print(f'  corpus complete at full target — skipping build')
        elif state['complete_shippable'] and not FORCE_CORPUS:
            print(f'  corpus is "complete enough" ({state["seqs"]} >= {state["shippable_seqs"]} '
                  f'shippable threshold = 60% of {CORPUS_TARGET_SEQS}). Skipping further build '
                  f'— Cell 3 trains on max-rows worth of windows, not the whole corpus.')
            print('  (set FORCE_CORPUS=True in Cell 1 to push to the full target)')
        else:
            os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
            os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
            local_out = f'/content/{tag}_corpus'
            sync_every = 1 if tag == 'q1p5' else 2
            cmd = [
                sys.executable, 'colab/mega_calibrate.py',
                '--model', meta['model_id'],
                '--max-sequences', str(CORPUS_TARGET_SEQS),
                '--max-tokens', '2048',
                '--batch-size', str(batch),
                '--capture-layer', str(meta['capture_layer']),
                '--topk-logits', '100',
                '--lm-head-chunk-tokens', str(lm_head_chunk),
                '--shard-size', '16',
                '--sync-dir', meta['corpus_dir'],
                '--sync-every', str(sync_every),
                '--delete-local-after-sync',
                '--out', local_out,
            ]
            if load_4bit:
                cmd.append('--load-4bit')
            try:
                run_calibration(cmd, tag=tag, local_out=local_out, sync_dir=meta['corpus_dir'])
            except TimeoutError as e:
                if tag in OPTIONAL_TARGETS and SKIP_OPTIONAL_TARGET_ON_CALIBRATION_STALL:
                    print(f'  optional target {tag} stalled: {e}', flush=True)
                    print(f'  disabling {tag} for downstream cells; q3b artifacts remain usable.', flush=True)
                    t['enabled'] = False
                    continue
                raise
        assert os.path.exists(stats_path), f'missing activation stats for {tag}: {stats_path}'

        # AWQ per-tensor (original heuristic).
        if os.path.exists(meta['awq_tensor']) and not FORCE_AWQ:
            print(f'  per-tensor AWQ exists: {meta["awq_tensor"]}')
        else:
            run([sys.executable, 'tools/training/awq_calibrate.py',
                 '--stats', stats_path, '--out', meta['awq_tensor'],
                 '--alpha', '0.5', '--model', meta['model_id']])

        # AWQ per-channel adaptive (the Eagle6 lever).
        if os.path.exists(meta['awq_channel']) and not FORCE_AWQ:
            print(f'  per-channel AWQ exists: {meta["awq_channel"]}')
        else:
            run([sys.executable, 'colab/awq_per_channel_calibrate.py',
                 '--stats', stats_path, '--out', meta['awq_channel'],
                 '--mode', 'adaptive-alpha',
                 '--alpha-floor', '0.3', '--alpha-ceil', '0.7',
                 '--model', meta['model_id']])

        # Q2/IQ2 importance artifact (sub-2-bit future ship inventory).
        if os.path.exists(meta['q2_importance']) and not FORCE_AWQ:
            print(f'  Q2 importance exists: {meta["q2_importance"]}')
        else:
            run([sys.executable, 'colab/q2k_importance_calibrate.py',
                 '--stats', stats_path, '--out', meta['q2_importance'],
                 '--model', meta['model_id']])

        # Frozen baseline dump (token_embd + lm_head + output_norm at fp16/fp32).
        if os.path.exists(meta['frozen']) and not FORCE_FROZEN:
            print(f'  frozen baseline exists: {meta["frozen"]}')
        else:
            run([sys.executable, 'colab/build_qwen3b_frozen_hf.py',
                 '--model', meta['model_id'], '--out', meta['frozen']])
        assert os.path.exists(meta['frozen']), f'frozen missing: {meta["frozen"]}'

        # Stage frozen to /content for fast trainer reads (avoids Drive FUSE OOM).
        local_frozen = f'{LOCAL_FROZEN_DIR}/{tag}_frozen.npz'
        if not os.path.exists(local_frozen) or os.path.getsize(local_frozen) != os.path.getsize(meta['frozen']):
            print(f'  staging frozen to local: {local_frozen}')
            shutil.copyfile(meta['frozen'], local_frozen)
        meta['local_frozen'] = local_frozen
        print(f'  frozen local: {local_frozen} ({os.path.getsize(local_frozen)/1e9:.2f} GB)')
finally:
    _hb_stop.set()
    _hb_thread.join(timeout=2)

print('\nCell 2 complete.')


In [ ]:
# Cell 3 - Eagle6 training: short grid + one extra-long pass on the winner.
import os, json, subprocess, sys, time, gc
import torch

# ---- VRAM-tiered training params ----------------------------------------
# Colab Pro silently demotes A100 → L4 → T4 mid-session sometimes; auto-adapt
# instead of OOM'ing. Tiers calibrated to Qwen-3B fp16 trainer working set:
#   - H100 80GB:   plenty of headroom, big batch + long tokens
#   - A100 40GB:   batch 16, full 192-token windows
#   - L4 24GB:     batch 8, 128-token windows on the long pass
#   - T4 / G4 16GB: batch 4, 96-token windows, fewer grid specs to stay sane
# Honors a manual TRAIN_BATCH_OVERRIDE in env / globals for forced runs.
_force_batch = int(os.environ.get('TRAIN_BATCH_OVERRIDE', '0') or '0')
if VRAM_GB >= 70:
    train_tier = 'H100 80GB'
    TRAIN_BATCH = 24
    LONG_TOKENS_EFF = LONG_MAX_ROW_TOKENS         # full 192
elif VRAM_GB >= 35:
    train_tier = 'A100 40GB'
    TRAIN_BATCH = 16
    LONG_TOKENS_EFF = LONG_MAX_ROW_TOKENS         # full 192
elif VRAM_GB >= 20:
    train_tier = 'L4 24GB'
    TRAIN_BATCH = 8
    LONG_TOKENS_EFF = min(128, LONG_MAX_ROW_TOKENS)
else:
    train_tier = 'T4/G4 16GB'
    TRAIN_BATCH = 4
    LONG_TOKENS_EFF = min(96, LONG_MAX_ROW_TOKENS)
if _force_batch > 0:
    TRAIN_BATCH = _force_batch
    print(f'[train] TRAIN_BATCH_OVERRIDE={_force_batch} (manual)')

# On T4-class boxes, the 5-spec grid is too slow. Cut to 3 specs so the run
# fits in a reasonable wall-clock. The long-train pass on the winner still
# happens at full quality.
if VRAM_GB < 20:
    GRID_EFFECTIVE = [s for s in EAGLE6_GRID if s['num_blocks'] == 1 or s['family'] == 'e6_b2_lite']
    print(f'[train] T4-class detected; trimming grid from {len(EAGLE6_GRID)} → {len(GRID_EFFECTIVE)} specs')
else:
    GRID_EFFECTIVE = EAGLE6_GRID

print(f'[train] tier={train_tier} batch={TRAIN_BATCH} long_tokens={LONG_TOKENS_EFF} '
      f'grid_specs={len(GRID_EFFECTIVE)}')
TAU_DEPTH = 4

def spec_tag(spec):
    lr_tag = f"lr{spec['lr']:.0e}".replace('+', '').replace('-0', '-')
    rd = int(round(spec['residual_delta'] * 1000))
    cw = int(round(spec['calib_weight'] * 100))
    return (f"{spec['family']}_b{spec['num_blocks']}_h{spec['head_heads']}_"
            f"ff{int(spec['head_ff_mult']*10)}_{lr_tag}_rd{rd:03d}_cw{cw:02d}")

def gc_pre_subproc():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_one(target_tag, meta, spec, *, epochs, max_rows, max_row_tokens, suffix):
    tag = spec_tag(spec) + suffix
    ckpt_dir = f'{ARTIFACT_DIR}/checkpoints/{target_tag}_{tag}'
    head_path = f'{ckpt_dir}/head_final.safetensors'
    os.makedirs(ckpt_dir, exist_ok=True)
    if os.path.exists(head_path) and not FORCE_RETRAIN:
        print(f'  skip trained {target_tag}/{tag}: head_final exists')
        return tag, head_path
    print(f'\n  === train {target_tag}/{tag} epochs={epochs} rows={max_rows} tokens={max_row_tokens} batch={TRAIN_BATCH}')
    gc_pre_subproc()
    cmd = [
        sys.executable, 'colab/eagle5_train_pytorch.py',
        '--corpus-dir', meta['corpus_dir'],
        '--frozen', meta['local_frozen'],
        '--ckpt-dir', ckpt_dir,
        '--epochs', str(epochs),
        '--batch-size', str(TRAIN_BATCH),
        '--seq-len', '16',
        '--lr', str(spec['lr']),
        '--capture-layer', str(meta['capture_layer']),
        '--max-rows', str(max_rows),
        '--max-row-tokens', str(max_row_tokens),
        '--sparsity-head', 'off',
        '--seed', '0',
        '--calib-loss-weight', str(spec['calib_weight']),
        '--residual-delta-loss-weight', str(spec['residual_delta']),
        '--num-blocks', str(spec['num_blocks']),
        '--head-heads', str(spec['head_heads']),
        '--head-ff-mult', str(spec['head_ff_mult']),
        '--save-safetensors',
    ]
    if USE_TORCH_COMPILE and VRAM_GB >= 35:
        cmd.append('--compile')
    run(cmd)
    assert os.path.exists(head_path), f'trainer did not write {head_path}'
    return tag, head_path

def tau_one(target_tag, meta, tag, head_path):
    out_path = f'{ARTIFACT_DIR}/checkpoints/{target_tag}_{tag}/tau.json'
    if os.path.exists(out_path) and not FORCE_TAU:
        return out_path
    gc_pre_subproc()
    base_tps = BASELINE_DEC_TPS_Q3B if target_tag == 'q3b' else BASELINE_DEC_TPS_Q15
    run([
        sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
        '--ckpt', head_path,
        '--frozen', meta['local_frozen'],
        '--corpus', meta['corpus_dir'],
        '--out', out_path,
        '--depth', str(TAU_DEPTH),
        '--max-windows', str(EVAL_MAX_WINDOWS),
        '--max-row-tokens', str(LONG_TOKENS_EFF),
        '--base-tps', str(base_tps),
        '--w4a8-multiplier', str(W4A8_MULTIPLIER),
        '--spec-efficiency', str(SPEC_EFFICIENCY),
    ])
    return out_path

all_trained = {}    # target -> [(tag, head_path, tau_dict), ...]
long_trained = {}   # target -> (tag, head_path)

for t in TARGETS:
    if not t['enabled']:
        continue
    target_tag = t['tag']
    meta = TARGET_META[target_tag]
    if 'local_frozen' not in meta:
        # Cell 2 might have been skipped on re-run; restore.
        meta['local_frozen'] = f'{LOCAL_FROZEN_DIR}/{target_tag}_frozen.npz'
        assert os.path.exists(meta['local_frozen']), 'run Cell 2 first'
    print(f'\n================ TARGET {target_tag} ({meta["model_id"]}) ================')

    grid_results = []
    if RUN_SHORT_GRID:
        for spec in GRID_EFFECTIVE:
            tag, hp = train_one(target_tag, meta, spec,
                                 epochs=GRID_EPOCHS,
                                 max_rows=GRID_MAX_ROWS,
                                 max_row_tokens=GRID_MAX_ROW_TOKENS,
                                 suffix='_grid')
            tau_path = tau_one(target_tag, meta, tag, hp)
            with open(tau_path) as f:
                tau = json.load(f)
            tau['_spec'] = spec
            grid_results.append((tag, hp, tau))
    all_trained[target_tag] = grid_results

    # Pick the grid winner by projected_dec_tps then tau then depth1 accept.
    if RUN_LONG_TRAIN and grid_results:
        ranked = sorted(grid_results,
            key=lambda x: (x[2]['projection']['projected_dec_tps'],
                           x[2]['tau'],
                           x[2]['depth1_accept_rate']),
            reverse=True)
        winner_tag, _, winner_tau = ranked[0]
        winner_spec = dict(winner_tau['_spec'])
        # Override LR/residual_delta/calib_weight with the long-train recipe.
        winner_spec.update(LONG_TRAIN_OVERRIDES)
        print(f'\n  GRID WINNER: {winner_tag} projected={winner_tau["projection"]["projected_dec_tps"]:.1f}')
        print(f'  LONG-TRAIN OVERRIDES: {LONG_TRAIN_OVERRIDES}')
        long_tag, long_hp = train_one(target_tag, meta, winner_spec,
                                       epochs=LONG_EPOCHS,
                                       max_rows=LONG_MAX_ROWS,
                                       max_row_tokens=LONG_TOKENS_EFF,
                                       suffix='_long')
        long_trained[target_tag] = (long_tag, long_hp)

print('\nCell 3 complete.')
print('grid trained:', {k: len(v) for k, v in all_trained.items()})
print('long trained:', {k: v[0] for k, v in long_trained.items()})


In [ ]:
# Cell 4 - Tau + frontier on long-trained heads, plus in-notebook spec-decode
# acceptance simulation.
import os, json, sys, subprocess, gc
import numpy as np
import torch

frontier_results = {}

def frontier_one(target_tag, tag, head_path):
    meta = TARGET_META[target_tag]
    out_path = f'{ARTIFACT_DIR}/checkpoints/{target_tag}_{tag}/frontier.json'
    if os.path.exists(out_path) and not FORCE_FRONTIER:
        return out_path
    base_tps = BASELINE_DEC_TPS_Q3B if target_tag == 'q3b' else BASELINE_DEC_TPS_Q15
    gc.collect(); torch.cuda.empty_cache()
    run([
        sys.executable, 'colab/eagle5_frontier_policy.py',
        '--ckpt', head_path,
        '--frozen', meta['local_frozen'],
        '--corpus', meta['corpus_dir'],
        '--out', out_path,
        '--max-depth', str(FRONTIER_MAX_DEPTH),
        '--depths', '4,6,8,12',
        '--lattice-widths', '2,3,4',
        '--max-windows', str(EVAL_MAX_WINDOWS),
        '--max-row-tokens', str(LONG_MAX_ROW_TOKENS),
        '--base-tps', str(base_tps),
        '--w4a8-multiplier', str(W4A8_MULTIPLIER),
        '--spec-efficiency', str(SPEC_EFFICIENCY),
    ])
    return out_path

for target_tag, (long_tag, long_hp) in long_trained.items():
    print(f'\n=== frontier {target_tag} / {long_tag} ===')
    fp = frontier_one(target_tag, long_tag, long_hp)
    with open(fp) as f:
        frontier_results[target_tag] = json.load(f)
        frontier_results[target_tag]['_head_path'] = long_hp
        frontier_results[target_tag]['_tag'] = long_tag

for target_tag, payload in frontier_results.items():
    best = payload['policies']['best_deployable']
    print(f'  {target_tag}: kind={best["kind"]} projected={best["projected_dec_tps"]:.1f} '
          f'accept_per_verify={best["accepted_draft_tokens_per_verify"]:.2f}')

# ---- In-notebook spec-decode simulation ----------------------------------
# The Colab projection multiplies (W4A8 × spec_efficiency × (1 + accept/verify))
# but those multipliers are model-side estimates, not Mac-runtime measurements.
# Today's local bench showed the projection was off by 4x because the Qwen
# Rust runtime doesn't invoke the Eagle head at all (see Cell 5 port plan).
#
# This cell does the only thing Colab CAN honestly verify: empirical accept
# rate of the trained head against the corpus. If it's high here, the head
# is good and the Mac port is the only missing piece. If it's low here, the
# head needs better training before the port matters.

def simulate_spec_accept(target_tag, head_path, frozen, corpus_dir, n_windows=200, K=4):
    """Loads the trained head + frozen baseline, replays K-step drafts against
    n_windows corpus windows, returns per-step accept counts.

    Implementation note: leverages the same Eagle5Head module the trainer uses
    (in `colab/eagle5_train_pytorch.py`) and a tiny in-notebook reimpl of the
    verify loop that the deepseek_v2.rs runtime already has — just enough to
    project accept-per-verify empirically.
    """
    sys.path.insert(0, '/content/dismantle/colab')
    from eagle5_train_pytorch import build_head
    from pathlib import Path
    import pyarrow.parquet as pq
    import torch as _torch
    import numpy as _np
    device = 'cuda'
    head = build_head(Path(frozen), with_sparsity=False, device=device)
    # Re-load trained weights from safetensors.
    from safetensors.torch import load_file as _load_safe
    sd = _load_safe(head_path, device=device)
    head.load_state_dict(sd, strict=False)
    head.eval()

    shards = sorted(Path(corpus_dir).glob('shard_*.parquet'))
    accept_per_step = _np.zeros(K, dtype=_np.float64)
    total = 0
    for sp in shards:
        if total >= n_windows:
            break
        t = pq.read_table(sp, columns=['tokens'])
        for i in range(t.num_rows):
            if total >= n_windows:
                break
            toks = t['tokens'][i].as_py()
            if isinstance(toks, (bytes, bytearray)):
                toks = _np.frombuffer(toks, dtype=_np.int32).copy()
            else:
                toks = _np.asarray(toks, dtype=_np.int32)
            if toks.size < 32:
                continue
            # Use the middle of the sequence as both prefix and ground truth.
            start = max(0, toks.size // 2 - 16)
            ctx = toks[start:start + 16]
            gt = toks[start + 16:start + 16 + K]
            if gt.size < K:
                continue
            ctx_t = _torch.from_numpy(ctx.astype(_np.int64)).unsqueeze(0).to(device)
            # Crude greedy draft: feed ctx, sample top-1 K times, compare to gt.
            # We can't replicate the full Rust verify (KV cache + interleave)
            # here, but we can check ARGMAX agreement which is the upper-bound.
            with _torch.no_grad():
                cur = ctx_t
                drafts = []
                for _ in range(K):
                    out = head(cur)
                    logits = out['logits'] if isinstance(out, dict) else out
                    nxt = logits[0, -1].argmax().item()
                    drafts.append(nxt)
                    cur = _torch.cat([cur, _torch.tensor([[nxt]], device=device, dtype=cur.dtype)], dim=1)
            for k in range(K):
                if drafts[k] == int(gt[k]):
                    accept_per_step[k] += 1
                else:
                    break
            total += 1
    if total == 0:
        return {'windows': 0, 'accept_rate_per_step': [0.0]*K, 'mean_accepted': 0.0}
    rates = (accept_per_step / total).tolist()
    mean_accepted = float(sum(rates))
    return {'windows': total, 'accept_rate_per_step': rates, 'mean_accepted': mean_accepted}

if RUN_SIMULATION:
    sim_results = {}
    for target_tag, payload in frontier_results.items():
        head_path = payload['_head_path']
        meta = TARGET_META[target_tag]
        try:
            print(f'\n--- simulation: {target_tag} ---')
            sim = simulate_spec_accept(
                target_tag, head_path, meta['local_frozen'], meta['corpus_dir'],
                n_windows=300, K=4,
            )
            sim_results[target_tag] = sim
            print(f'  windows={sim["windows"]}  per-step accept={sim["accept_rate_per_step"]}  '
                  f'mean_accepted={sim["mean_accepted"]:.3f}')
        except Exception as e:
            print(f'  simulation FAILED for {target_tag}: {e}')
            sim_results[target_tag] = {'error': str(e)}
    sim_path = f'{ARTIFACT_DIR}/reconciliation_simulation.json'
    with open(sim_path, 'w') as f:
        json.dump(sim_results, f, indent=2, sort_keys=True)
    print(f'\nSim summary written to {sim_path}')

frontier_winner_path = f'{ARTIFACT_DIR}/reconciliation_frontier_winners.json'
with open(frontier_winner_path, 'w') as f:
    json.dump({
        'frontier_per_target': {k: {
            'tag': v['_tag'],
            'head_path': v['_head_path'],
            'best_deployable': v['policies']['best_deployable'],
            'runtime_hints': v.get('runtime_hints', {}),
        } for k, v in frontier_results.items()},
    }, f, indent=2, sort_keys=True)
print(f'\nfrontier winners → {frontier_winner_path}')

In [ ]:
# Cell 5 - Summary + LOCAL RUST PORT PLAN.
import os, json, time

summary_lines = [
    '# Qwen Reconciliation Pipeline Summary',
    '',
    f'Generated on `{GPU_NAME}` from `colab/qwen3b_reconciliation.ipynb`.',
    '',
    '## Mode',
    f'- MAX_QUALITY_MODE: `{MAX_QUALITY_MODE}`',
    f'- corpus target: `{CORPUS_TARGET_SEQS}` sequences',
    f'- grid epochs: `{GRID_EPOCHS}`, long-train epochs: `{LONG_EPOCHS}`',
    f'- long max-row-tokens: `{LONG_MAX_ROW_TOKENS}`',
    '',
    '## Long-trained Eagle6 winners',
]
for tt, (tag, hp) in long_trained.items():
    fr = frontier_results.get(tt, {}).get('policies', {}).get('best_deployable', {})
    summary_lines.append(
        f'- **{tt}**: `{tag}` → projected `{fr.get("projected_dec_tps", "?"):.1f}` dec_tps '
        f'(kind=`{fr.get("kind", "?")}`, accept/verify=`{fr.get("accepted_draft_tokens_per_verify", "?"):.2f}`)'
    )
    summary_lines.append(f'  head: `{hp}`')

summary_lines.extend([
    '',
    '## In-notebook spec-decode simulation',
    'Empirical accept-per-verify against held-out corpus windows. This is the',
    'truthful upper bound — the Mac runtime can only do this well or worse.',
    '',
])
try:
    for tt, sim in sim_results.items():
        if 'error' in sim:
            summary_lines.append(f'- {tt}: simulation FAILED — `{sim["error"]}`')
        else:
            summary_lines.append(
                f'- **{tt}**: mean_accepted=`{sim["mean_accepted"]:.3f}` over {sim["windows"]} windows '
                f'(per-step: {[round(r,3) for r in sim["accept_rate_per_step"]]})'
            )
except NameError:
    summary_lines.append('- (simulation skipped — RUN_SIMULATION=False)')

summary_lines.extend([
    '',
    '## What you need to know',
    '',
    'These trained heads will not deliver any dec_tps speedup on the Mac yet.',
    'Reason: spec-decode is wired into `crates/dismantle-core/src/model/deepseek_v2.rs` only.',
    '`qwen_dense.rs` has **zero** references to Eagle5 — the trained head is silently',
    'ignored at runtime on Qwen-3B/1.5B. Port plan: `docs/eagle5_qwen_port_plan.md`.',
    '',
    '## Local handoff commands',
    '',
    '```bash',
    '# Download the long-trained head(s) from Drive to the Mac project tree:',
])
for tt, (tag, hp) in long_trained.items():
    summary_lines.append(
        f'#   {tt}: gdrive→checkpoints/eagle6_{tt}_lr3e-3_long/head_final.safetensors'
    )
    summary_lines.append(f'#   (source path on Drive: {hp})')
summary_lines.extend([
    '```',
    '',
    '## Next session = local Rust port (NOT Colab)',
    'See `docs/eagle5_qwen_port_plan.md` for the file/line refs in deepseek_v2.rs',
    'that need to be ported to qwen_dense.rs, plus the validation gates.',
    'Expected effort: 2-4 focused days. The trained heads in this notebook are',
    'the *inventory* that port will consume.',
])

summary = '\n'.join(summary_lines) + '\n'
summary_path = f'{ARTIFACT_DIR}/reconciliation_summary.md'
with open(summary_path, 'w') as f:
    f.write(summary)
print(summary)
print(f'wrote {summary_path}')